# Incident Commander — Reproducible training artifact for OpenEnv Hackathon judges

**This notebook reproduces our headline training result: SFT-on-oracle.**

| Metric | Expected (from committed `training/sft_metrics.json`) |
|---|---:|
| Macro-mean delta | **+0.122** (pre 0.693 → post 0.815) |
| Hard-task delta | **+0.548** (0.347 → 0.895) |
| Easy-task delta | −0.195 (intentional trade — see blog) |
| Total runtime | ~25 minutes on a T4 |
| Regression gate threshold | +0.05 macro |

Re-running on a different T4 should produce numbers within **±0.05** of the
committed reference (GPU non-determinism in matmul kernels and sampling RNG
state introduces small drift; the magnitude and direction of the gains are
stable).

## Why this notebook and not `train_colab.ipynb` or `train_grpo_colab.ipynb`?

| Notebook | Stage | Why we don't run it as the judges' artifact |
|---|---|---|
| `train_sft_colab.ipynb` | **SFT-on-oracle** ✅ | **This one.** Supervised on a deterministic teacher; reproducible. |
| `train_colab.ipynb` | RFT polish on SFT-warm | Produced **+0.190** on HF Jobs T4 but **−0.793** on a separate Colab T4 — known on-policy variance. Inspect the code, don't re-run for verification. |
| `train_grpo_colab.ipynb` | Original GRPO attempt | Regressed by −0.356. Preserved as a negative-result artifact. |

The full story (5 attempts, 2 wins, 3 regressions) is in the [`blog.md`](../blog.md). This notebook is the data path that produced the committed `training/sft_metrics.json` headline numbers.

---


# Incident Commander — SFT-on-oracle warm-start

This notebook is the judges' reproducible artifact for Phase 2 of the
Incident Commander training recovery plan. It downloads the repository,
installs the GPU training extras, generates ~540 labeled `(observation,
action)` pairs from deterministic scripted oracles, fine-tunes a LoRA
adapter on `unsloth/Llama-3.2-3B-Instruct` via TRL `SFTTrainer`, evaluates
pre/post on all 3 tasks, plots loss + reward curves, and (optionally)
pushes the adapter to HuggingFace Hub.

Target hardware: T4 / T4-medium. Total runtime: ~45 minutes.

**Prerequisites**
* HuggingFace account with token (read **+ write** scope) saved in Colab
  Secrets as `HF_TOKEN`.
* No API key needed — all generation runs in-process via Unsloth.

**What the regression gate enforces** — `train_sft.py` refuses to save the
adapter unless the post-training macro-mean reward beats the baseline by
≥ +0.10 *and* no individual rubric component (RCA / mitigation /
post-mortem) regresses by more than 0.005. If the gate trips, the metrics
JSON is still written so you can inspect the failure.

## 1. Hardware check

Confirm the runtime has a GPU. If `nvidia-smi` errors, switch the Colab
runtime to GPU (Runtime → Change runtime type → GPU).

In [ ]:
!nvidia-smi

## 2. Clone the Incident Commander repo

The HF Space mirror is the source of truth — same files as the deployed
env. Replace `vasubhrdwj` with your HF username if you forked it.

In [ ]:
import os, sys

# Where to clone the HF Space repo inside Colab. The Space's contents ARE the
# package root (pyproject.toml is at the top level), so we cd into the cloned
# directory directly — NOT into a 'incident_commander' subfolder.
REPO_DIR = '/content/incident-commander-openenv'
if not os.path.isdir(REPO_DIR):
    !git clone https://huggingface.co/spaces/vasubhrdwj/incident-commander-openenv $REPO_DIR

# Use %cd (IPython magic) so the directory change persists across cells.
# !cd would only affect the single bash subprocess and reset for the next cell.
%cd $REPO_DIR
!ls

# Sanity check — pyproject.toml MUST be visible here for the next cell's
# `pip install -e '.[training]'` to work.
assert os.path.exists('pyproject.toml'), (
    f'pyproject.toml not found in {os.getcwd()} — clone may have failed'
)
print('OK — repo cloned, cwd has pyproject.toml')


## 3. Install training dependencies

Installs Unsloth + TRL + bitsandbytes + matplotlib (the `[training]`
extra). Takes ~3-5 min on a T4.

In [ ]:
!pip install -q -e '.[training]'

## 4. HuggingFace login

Pulls your token from Colab Secrets — set `HF_TOKEN` in the left sidebar
first. Token must have **write** scope if you plan to push the adapter in
step 9.

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'Set HF_TOKEN in Colab Secrets (left sidebar) first.'
login(token=HF_TOKEN, add_to_git_credential=True)
print('logged in')

## 5. Build the SFT-on-oracle dataset

Generates ~540 labeled `(observation, action)` pairs by running the
deterministic scripted oracles in-process across 30 seeds × 3 tasks. Pure
CPU — runs in <1 minute.

In [ ]:
# IMPORTANT: train on medium + hard only.
#
# Training on all 3 tasks (the default if you omit --tasks) was Attempt 3
# in our journey and regressed by -0.094 macro: SFT memorized fixed action
# sequences on the easy task (where the prompt was already at 0.96 ceiling)
# and the hard-task postmortem JSON parse-failed on 2 of 3 seeds.
#
# Dropping easy from training data and trimming the postmortem schema is
# what made Attempt 4 work (+0.122 macro, +0.548 on hard). See blog.md
# for the full journey.
!python -m incident_commander.training.build_sft_dataset \
    --seeds 30 \
    --tasks medium_third_party_attribution hard_silent_data_corruption \
    --output sft_oracle.jsonl


## 6. Train

Runs pre-eval, fine-tunes the LoRA adapter for 1 epoch, runs post-eval,
then enforces the regression gate before saving. Takes ~25-30 min on a
T4 with default settings.

In [ ]:
!python -m incident_commander.training.train_sft \
    --base-model unsloth/Llama-3.2-3B-Instruct \
    --dataset sft_oracle.jsonl \
    --output-dir ./ic-sft-oracle \
    --metrics-json sft_metrics.json \
    --epochs 1 --lr 1e-4 \
    --batch-size 2 --grad-accum 4 \
    --lora-r 16 --lora-alpha 32 \
    --eval-seeds 3 \
    --min-improvement 0.05


## 7. Plot metrics

Produces three PNGs:
* `sft_loss.png` — training loss vs logged step.
* `sft_pre_post.png` — mean reward per task, baseline vs trained.
* `sft_components.png` — six rubric components per task, pre vs post.

In [ ]:
!python -m incident_commander.training.plot_metrics --mode sft \
    --metrics sft_metrics.json --out assets/

## 8. Inline preview

In [ ]:
from IPython.display import Image, display
for png in ('sft_loss.png', 'sft_pre_post.png', 'sft_components.png'):
    path = f'assets/{png}'
    if os.path.isfile(path):
        print(path)
        display(Image(path))
    else:
        print(f'(missing: {path})')

## 9. Push the LoRA adapter to HF Hub (optional)

Skip if the gate didn't pass — the adapter wasn't saved in that case.
Replace `vasubhrdwj` with your HF username before running.

In [ ]:
import json
summary = json.load(open('sft_metrics.json'))
if summary.get('saved_adapter') and summary.get('gate_pass'):
    !huggingface-cli upload vasubhrdwj/incident-commander-sft-lora ./ic-sft-oracle . \
        --repo-type model --commit-message 'SFT-on-oracle adapter (Phase 2)'
else:
    print(f"adapter not pushed; gate_pass={summary.get('gate_pass')} "
          f"reason={summary.get('gate_reason')!r}")

## 10. Headline numbers

Final summary printout — copy these into the README table.

In [ ]:
import json
s = json.load(open('sft_metrics.json'))
pre = s['pre_eval']
post = s['post_eval']
print(f"base model:        {s['base_model']}")
print(f"train wall-time:   {s.get('train_seconds', 0):.1f}s")
print(f"gate passed:       {s.get('gate_pass')}  ({s.get('gate_reason') or 'n/a'})")
print(f"adapter saved:     {s.get('saved_adapter')}")
print()
print(f"{'task':<35s} {'pre':>7s} {'post':>7s} {'delta':>8s}")
for task in pre:
    p = pre[task]['mean']
    q = post[task]['mean']
    print(f"{task:<35s} {p:>7.3f} {q:>7.3f} {q - p:>+8.3f}")
macro_pre = sum(t['mean'] for t in pre.values()) / len(pre)
macro_post = sum(t['mean'] for t in post.values()) / len(post)
print(f"{'macro-mean':<35s} {macro_pre:>7.3f} {macro_post:>7.3f} {macro_post - macro_pre:>+8.3f}")